# Détection des maladies de la tomate par CNN : du laboratoire au champ

**Projet de fin de module Deep Learning, cohorte 3 AI Hub**
Groupe : *[noms]* · Rendu : 10/11/2026

**Question centrale** : comment construire un modèle qui fonctionne vraiment dans un champ ?

| Jeu | Rôle | Conditions |
|---|---|---|
| PlantVillage (tomate) | apprendre | laboratoire, fond uniforme |
| PlantDoc (tomate) | vérifier / adapter | terrain, fonds chargés |

**Plan du notebook**
0. Installation et données
1. Exploration des données
2. Pipeline de données et augmentations
3. Baseline MLP
4. CNN from scratch : comparaison de variantes
5. Test de robustesse PlantVillage → PlantDoc et stratégies
6. Analyse des erreurs
7. Explicabilité (Grad-CAM)
8. Tableau récapitulatif

> ⚙️ Sur Colab : *Exécution > Modifier le type d'exécution > GPU T4*.
> Les cellules marquées ✍️ sont à compléter par le groupe (analyses, hypothèses).

## 0. Installation et données

In [ ]:
import os, sys, subprocess
from pathlib import Path

# 👉 Remplacez par l'URL de VOTRE dépôt GitHub
REPO_URL = "https://github.com/VOTRE_COMPTE/plant-disease-cnn.git"

if not Path("scripts/prepare_data.py").exists():          # on est sur Colab / Kaggle
    if not Path("plant-disease-cnn").exists():
        subprocess.run(["git", "clone", REPO_URL, "plant-disease-cnn"], check=True)
    os.chdir("plant-disease-cnn")
print("Dossier de travail :", os.getcwd())

In [ ]:
# Téléchargement (~2,4 Go, quelques minutes) puis construction du manifest.
# --pv-max-per-class limite PlantVillage pour accélérer l'entraînement sur Colab gratuit.
!python scripts/download_data.py --out data/raw
!python scripts/prepare_data.py --raw data/raw --out data/manifest.csv --pv-max-per-class 1500

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
from PIL import Image
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_fscore_support, recall_score, f1_score)
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
IMG_SIZE = 128
BATCH = 32
EPOCHS = 30            # l'early stopping arrête généralement avant
AUTOTUNE = tf.data.AUTOTUNE
keras.utils.set_random_seed(SEED)

CLASSES = ["bacterial_spot", "early_blight", "late_blight", "leaf_mold",
           "septoria_leaf_spot", "yellow_leaf_curl_virus", "mosaic_virus", "healthy"]
N_CLASSES = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
HEALTHY = CLASS_TO_IDX["healthy"]

os.makedirs("models", exist_ok=True); os.makedirs("results", exist_ok=True)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| GPU :", tf.config.list_physical_devices("GPU"))

manifest = pd.read_csv("data/manifest.csv")
manifest["y"] = manifest["label"].map(CLASS_TO_IDX)
split = {s: d.reset_index(drop=True) for s, d in manifest.groupby("split")}
{s: len(d) for s, d in split.items()}

## 1. Exploration des données

In [ ]:
counts = manifest.pivot_table(index="label", columns="split", values="path", aggfunc="count", fill_value=0)
counts = counts.reindex(CLASSES)[["pv_train", "pv_val", "pv_test", "pd_train", "pd_val", "pd_test"]]
display(counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, src, title in zip(axes, ["pv", "pd"], ["PlantVillage (tomate)", "PlantDoc (tomate)"]):
    c = manifest[manifest.source == src]["label"].value_counts().reindex(CLASSES)
    ax.barh(CLASSES, c.values)
    ax.set_title(f"{title} : {c.sum()} images"); ax.invert_yaxis()
    ratio = c.max() / c.min()
    ax.set_xlabel(f"déséquilibre max/min = {ratio:.1f}")
plt.tight_layout(); plt.show()

In [ ]:
def show_examples(source, n_per_class=4, seed=0):
    df = manifest[manifest.source == source]
    fig, axes = plt.subplots(N_CLASSES, n_per_class, figsize=(2.2 * n_per_class, 2.2 * N_CLASSES))
    for i, c in enumerate(CLASSES):
        sample = df[df.label == c].sample(n_per_class, random_state=seed)
        for j, p in enumerate(sample.path):
            ax = axes[i, j]
            ax.imshow(Image.open(p).convert("RGB")); ax.axis("off")
            if j == 0: ax.set_title(c, fontsize=9, loc="left")
    fig.suptitle("PlantVillage" if source == "pv" else "PlantDoc"); plt.tight_layout(); plt.show()

show_examples("pv")
show_examples("pd")

In [ ]:
# Taille des images d'origine : PlantVillage est homogène, PlantDoc très variable
rows = []
for src in ["pv", "pd"]:
    for p in manifest[manifest.source == src].sample(300, random_state=SEED).path:
        with Image.open(p) as im:
            rows.append({"source": src, "w": im.width, "h": im.height})
sizes = pd.DataFrame(rows)
display(sizes.groupby("source").describe().round(0))

✍️ **Observations à rédiger** (nombre de classes, déséquilibre, différences visuelles PV/PD, qualité, classes les plus rares dans PlantDoc…)

*Notes sur le découpage (voir `scripts/prepare_data.py`)* :
- PlantVillage : découpage **groupé par feuille** (suffixe du nom de fichier) pour éviter qu'une même feuille photographiée plusieurs fois se retrouve en train et en test.
- PlantDoc : doublons exacts supprimés (hash MD5) ; test refait par tirage stratifié car le test officiel ne contient qu'environ 70 images tomate.
- `pd_test` n'est **jamais** utilisé pour entraîner ni pour choisir un modèle.

## 2. Pipeline de données et augmentations

In [ ]:
def load_image(path, y):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(tf.clip_by_value(img, 0, 255), tf.uint8), y      # uint8 pour un cache léger

# Augmentation "classique" (rotations, retournements, recadrage, luminosité, contraste)
augment_basic = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2, fill_mode="reflect"),
    layers.RandomZoom((-0.3, 0.1), fill_mode="reflect"),   # valeur négative = zoom avant = recadrage
    layers.RandomTranslation(0.1, 0.1, fill_mode="reflect"),
    layers.RandomBrightness(0.3),                          # images dans [0, 255]
    layers.RandomContrast(0.3),
], name="augment_basic")

def leaf_mask(img):
    # Masque approximatif de la feuille : le fond PlantVillage est peu saturé (gris/beige).
    hsv = tf.image.rgb_to_hsv(tf.cast(img, tf.float32) / 255.0)
    m = tf.cast(hsv[..., 1] > 0.25, tf.float32)[None, ..., None]
    m = tf.nn.avg_pool2d(m, ksize=9, strides=1, padding="SAME")       # adoucit les bords
    return tf.clip_by_value(m[0] * 1.5, 0, 1)                          # (H, W, 1)

# Banque de fonds : images de pd_train (aucune étiquette utilisée, jamais pd_test).
# Alternative sans données de terrain : BG_SOURCE = "noise".
BG_SOURCE = "plantdoc"
if BG_SOURCE == "plantdoc":
    BG_BANK = np.stack([np.array(Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)))
                        for p in split["pd_train"].path]).astype("uint8")
    BG_BANK = tf.constant(BG_BANK)

def random_background(img, p=0.6):
    img = tf.cast(img, tf.float32)
    if BG_SOURCE == "plantdoc":
        bg = tf.cast(tf.gather(BG_BANK, tf.random.uniform([], 0, tf.shape(BG_BANK)[0], tf.int32)), tf.float32)
        bg = tf.image.random_flip_left_right(bg)
    else:
        bg = tf.random.uniform((IMG_SIZE, IMG_SIZE, 3), 0, 255)
        bg = tf.nn.avg_pool2d(bg[None], 15, 1, "SAME")[0]
    m = leaf_mask(img)
    mixed = m * img + (1 - m) * bg
    return tf.where(tf.random.uniform([]) < p, mixed, img)

def make_ds(df, training=False, augment=None, background=False, repeat=False):
    ds = tf.data.Dataset.from_tensor_slices((df.path.values, df.y.values.astype("int32")))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    if repeat:
        ds = ds.repeat()
    if background:
        ds = ds.map(lambda x, y: (random_background(x), y), num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH)
    if augment is not None:
        ds = ds.map(lambda x, y: (tf.clip_by_value(augment(x, training=True), 0, 255), y),
                    num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

In [ ]:
# Vérification visuelle : originaux / augmentation classique / changement de fond
sample = split["pv_train"].sample(6, random_state=1)
raw = np.stack([load_image(p, 0)[0].numpy() for p in sample.path]).astype("float32")
aug = np.clip(augment_basic(raw, training=True).numpy(), 0, 255)
bgr = np.stack([random_background(x, p=1.0).numpy() for x in raw])

fig, axes = plt.subplots(3, 6, figsize=(14, 7))
for row, (imgs, name) in enumerate([(raw, "original"), (aug, "augment_basic"), (bgr, "nouveau fond")]):
    for j in range(6):
        axes[row, j].imshow(imgs[j].astype("uint8")); axes[row, j].axis("off")
    axes[row, 0].set_title(name, loc="left")
plt.tight_layout(); plt.show()

✍️ Le masque de feuille est-il correct ? Sur quelles images échoue-t-il (feuilles jaunies, fond coloré) ? Ajustez le seuil de saturation si besoin.

## 3. Outils communs : entraînement et évaluation

**Métrique principale.** Une maladie non détectée (faux négatif) coûte plus cher qu'une fausse alerte : la maladie se propage et la récolte est perdue. On suit donc :
- le **rappel macro** (moyenne des rappels par classe, insensible au déséquilibre) comme métrique principale ;
- le **rappel « malade »** : part des feuilles malades qui ne sont *pas* classées saines (aucune maladie ratée) ;
- le **taux de fausse alerte** : part des feuilles saines classées malades ;
- précision, rappel et F1 **par classe**, et F1 macro.

✍️ Adaptez cette justification dans le rapport.

In [ ]:
RESULTS = []        # une ligne par (modèle, jeu de test)
PREDS = {}          # prédictions conservées pour l'analyse d'erreurs

def class_weights(df):
    w = compute_class_weight("balanced", classes=np.arange(N_CLASSES), y=df.y.values)
    return dict(enumerate(w))

def train(model, train_ds, val_ds, epochs=EPOCHS, lr=1e-3, class_weight=None, steps_per_epoch=None, patience=6):
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
    ]
    hist = model.fit(train_ds, validation_data=val_ds, epochs=epochs, class_weight=class_weight,
                     steps_per_epoch=steps_per_epoch, callbacks=callbacks, verbose=2)
    model.save(f"models/{model.name}.keras")
    return hist

def plot_history(hist, title):
    h = hist.history
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    for ax, k in zip(axes, ["loss", "accuracy"]):
        ax.plot(h[k], label="train"); ax.plot(h["val_" + k], label="validation")
        ax.set_title(f"{title} : {k}"); ax.set_xlabel("époque"); ax.legend()
    plt.tight_layout(); plt.show()

def evaluate(model, test_split, tag=None, verbose=True):
    df = split[test_split]
    probs = model.predict(make_ds(df), verbose=0)
    y_true, y_pred = df.y.values, probs.argmax(1)
    sick = y_true != HEALTHY
    row = {
        "modèle": tag or model.name, "test": test_split, "n": len(df),
        "accuracy": (y_true == y_pred).mean(),
        "rappel_macro": recall_score(y_true, y_pred, average="macro", labels=range(N_CLASSES), zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", labels=range(N_CLASSES), zero_division=0),
        "rappel_malade": (y_pred[sick] != HEALTHY).mean(),
        "fausse_alerte": (y_pred[~sick] != HEALTHY).mean() if (~sick).any() else np.nan,
    }
    RESULTS.append(row)
    PREDS[(row["modèle"], test_split)] = (y_true, y_pred, probs)
    if verbose:
        print(f"== {row['modèle']} sur {test_split} ==")
        print(classification_report(y_true, y_pred, labels=range(N_CLASSES), target_names=CLASSES,
                                    digits=3, zero_division=0))
    return row

def results_table():
    return pd.DataFrame(RESULTS).drop_duplicates(["modèle", "test"], keep="last").round(3)

## 4. Baseline MLP (images aplaties)

Le MLP ignore la structure spatiale de l'image : chaque pixel est une entrée indépendante. Son rôle est de montrer pourquoi le CNN est nécessaire (performances, nombre de paramètres, généralisation).

In [ ]:
def build_mlp(input_size=64):
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = layers.Resizing(input_size, input_size)(inp)      # 64x64x3 = 12 288 entrées
    x = layers.Rescaling(1 / 255.0)(x)
    x = layers.Flatten()(x)
    for units in (512, 256):
        x = layers.Dense(units, activation="relu")(x)
        x = layers.Dropout(0.5)(x)
    out = layers.Dense(N_CLASSES, activation="softmax")(x)
    return keras.Model(inp, out, name="mlp")

pv_train_ds = make_ds(split["pv_train"], training=True)
pv_val_ds = make_ds(split["pv_val"])
cw_pv = class_weights(split["pv_train"])

mlp = build_mlp()
mlp.summary()
hist_mlp = train(mlp, pv_train_ds, pv_val_ds, class_weight=cw_pv)
plot_history(hist_mlp, "MLP")
evaluate(mlp, "pv_test"); evaluate(mlp, "pd_test");

## 5. CNN conçu from scratch : comparaison de variantes

In [ ]:
def build_cnn(filters=(32, 64, 128), batchnorm=True, dropout=0.4, name="cnn"):
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(1 / 255.0)(inp)
    for b, f in enumerate(filters):
        for k in range(2):
            last = (b == len(filters) - 1) and (k == 1)
            x = layers.Conv2D(f, 3, padding="same", use_bias=not batchnorm)(x)
            if batchnorm:
                x = layers.BatchNormalization()(x)
            x = layers.Activation("relu", name="last_conv" if last else None)(x)   # cible du Grad-CAM
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.2)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(N_CLASSES, activation="softmax")(x)
    return keras.Model(inp, out, name=name)

VARIANTS = {
    "cnn_A_sans_bn": dict(filters=(32, 64, 128), batchnorm=False),
    "cnn_B_bn":      dict(filters=(32, 64, 128), batchnorm=True),
    "cnn_C_bn_prof": dict(filters=(32, 64, 128, 256), batchnorm=True),   # plus profond
}
models, histories = {}, {}
for name, kw in VARIANTS.items():
    keras.utils.set_random_seed(SEED)
    models[name] = build_cnn(name=name, **kw)
    print(f"\n### {name} : {models[name].count_params():,} paramètres")
    histories[name] = train(models[name], pv_train_ds, pv_val_ds, class_weight=cw_pv)
    plot_history(histories[name], name)
    evaluate(models[name], "pv_test", verbose=False)

print(f"MLP : {mlp.count_params():,} paramètres")
results_table()

In [ ]:
# Courbes de validation des variantes superposées
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for name, h in histories.items():
    axes[0].plot(h.history["val_loss"], label=name)
    axes[1].plot(h.history["val_accuracy"], label=name)
axes[0].set_title("val_loss"); axes[1].set_title("val_accuracy")
for ax in axes: ax.set_xlabel("époque"); ax.legend()
plt.tight_layout(); plt.show()

✍️ **Analyse** : surapprentissage (écart train/validation) ? Effet de la batch normalization sur la vitesse et la stabilité ? Effet de la profondeur ? MLP vs CNN (performance et nombre de paramètres) ?

Choisissez la meilleure variante pour la suite (`BEST_VARIANT`).

In [ ]:
BEST_VARIANT = "cnn_B_bn"     # ✍️ à ajuster selon vos résultats
BEST_KW = VARIANTS[BEST_VARIANT]

## 6. Test de robustesse : PlantVillage → PlantDoc (cœur du projet)

### 6.1 Mesure du fossé laboratoire / terrain

In [ ]:
# E0 : la meilleure variante, entraînée uniquement sur PlantVillage, sans augmentation
e0 = models[BEST_VARIANT]
evaluate(e0, "pd_test", tag="E0_pv_seul")
evaluate(e0, "pv_test", tag="E0_pv_seul", verbose=False)

t = results_table(); t = t[t["modèle"] == "E0_pv_seul"].set_index("test")
print("Chute du rappel macro PV → PD :",
      round(t.loc["pv_test", "rappel_macro"] - t.loc["pd_test", "rappel_macro"], 3))

### 6.2 Stratégies pour réduire l'écart

In [ ]:
# E1 : PlantVillage + augmentation classique
keras.utils.set_random_seed(SEED)
e1 = build_cnn(name="E1_pv_augment", **BEST_KW)
h = train(e1, make_ds(split["pv_train"], training=True, augment=augment_basic), pv_val_ds, class_weight=cw_pv)
plot_history(h, e1.name)
evaluate(e1, "pv_test", verbose=False); evaluate(e1, "pd_test");

In [ ]:
# E2 : PlantVillage + augmentation classique + changement de fond
keras.utils.set_random_seed(SEED)
e2 = build_cnn(name="E2_pv_augment_fond", **BEST_KW)
h = train(e2, make_ds(split["pv_train"], training=True, augment=augment_basic, background=True),
          pv_val_ds, class_weight=cw_pv)
plot_history(h, e2.name)
evaluate(e2, "pv_test", verbose=False); evaluate(e2, "pd_test");

In [ ]:
# E3 : ré-entraînement (fine-tuning) du meilleur modèle PlantVillage sur pd_train
# On part des poids appris sur PV et on continue avec un learning rate faible.
SOURCE_MODEL = e2                      # ✍️ ou e1 / e0 selon vos résultats
pd_train_ds = make_ds(split["pd_train"], training=True, augment=augment_basic)
pd_val_ds = make_ds(split["pd_val"])
cw_pd = class_weights(split["pd_train"])

clone = keras.models.clone_model(SOURCE_MODEL); clone.set_weights(SOURCE_MODEL.get_weights())
e3 = keras.Model(clone.inputs, clone.outputs, name="E3_finetune_pd")   # même réseau, nouveau nom
# Option : geler les premiers blocs pour limiter le surapprentissage sur peu d'images
FREEZE_FIRST_N_CONV = 0
for layer in [l for l in e3.layers if isinstance(l, layers.Conv2D)][:FREEZE_FIRST_N_CONV]:
    layer.trainable = False
h = train(e3, pd_train_ds, pd_val_ds, lr=1e-4, class_weight=cw_pd, patience=8)
plot_history(h, e3.name)
evaluate(e3, "pv_test", verbose=False); evaluate(e3, "pd_test");

In [ ]:
# E4 : entraînement mixte PlantVillage + PlantDoc (from scratch)
# PlantDoc étant ~15x plus petit, on échantillonne 50/50 pour qu'il ne soit pas noyé.
keras.utils.set_random_seed(SEED)
mix_train = tf.data.Dataset.sample_from_datasets(
    [make_ds(split["pv_train"], training=True, augment=augment_basic, background=True, repeat=True).unbatch(),
     make_ds(split["pd_train"], training=True, augment=augment_basic, repeat=True).unbatch()],
    weights=[0.5, 0.5], seed=SEED).batch(BATCH).prefetch(AUTOTUNE)
steps = len(split["pv_train"]) // BATCH

e4 = build_cnn(name="E4_mixte_pv_pd", **BEST_KW)
h = train(e4, mix_train, pd_val_ds, steps_per_epoch=steps, patience=8)   # validation sur le terrain
plot_history(h, e4.name)
evaluate(e4, "pv_test", verbose=False); evaluate(e4, "pd_test");

In [ ]:
# E5 (référence) : CNN entraîné uniquement sur pd_train. PlantVillage aide-t-il vraiment ?
keras.utils.set_random_seed(SEED)
e5 = build_cnn(name="E5_pd_seul", **BEST_KW)
h = train(e5, pd_train_ds, pd_val_ds, class_weight=cw_pd, patience=10)
plot_history(h, e5.name)
evaluate(e5, "pv_test", verbose=False); evaluate(e5, "pd_test");

In [ ]:
t = results_table()
pivot = t.pivot(index="modèle", columns="test", values=["rappel_macro", "f1_macro", "rappel_malade"])
display(pivot)

pd_only = t[t.test == "pd_test"].set_index("modèle")
ax = pd_only[["rappel_macro", "f1_macro", "rappel_malade"]].plot.barh(figsize=(9, 5))
ax.set_title("Performance sur PlantDoc (test terrain)"); ax.set_xlim(0, 1); ax.invert_yaxis()
plt.tight_layout(); plt.show()

✍️ **Analyse** : quelle stratégie réduit le plus l'écart ? L'augmentation seule suffit-elle ? Le fine-tuning fait-il oublier PlantVillage (performance sur `pv_test`) ? PlantVillage aide-t-il par rapport à E5 ?

⚠️ Avec environ 185 images en `pd_test`, une différence de 2 à 3 points n'est pas forcément significative. Pour aller plus loin : relancer avec 2 ou 3 graines différentes et rapporter moyenne ± écart-type.

## 7. Analyse des erreurs

In [ ]:
BEST_MODEL = e3          # ✍️ meilleur modèle sur pd_test
best_tag = BEST_MODEL.name

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, test in zip(axes, ["pv_test", "pd_test"]):
    y_true, y_pred, _ = PREDS[(best_tag, test)]
    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=range(N_CLASSES), normalize="true"),
                           display_labels=CLASSES).plot(ax=ax, cmap="Blues", values_format=".2f",
                                                        colorbar=False, xticks_rotation=45)
    ax.set_title(f"{best_tag} : {test} (normalisée par ligne = rappel)")
plt.tight_layout(); plt.show()

# Rappel par classe, labo vs terrain
rec = {}
for test in ["pv_test", "pd_test"]:
    y_true, y_pred, _ = PREDS[(best_tag, test)]
    rec[test] = precision_recall_fscore_support(y_true, y_pred, labels=range(N_CLASSES), zero_division=0)[1]
pd.DataFrame(rec, index=CLASSES).plot.barh(figsize=(8, 4), title=f"Rappel par classe : {best_tag}").invert_yaxis()
plt.tight_layout(); plt.show()

In [ ]:
def show_errors(tag, test, n=12):
    df = split[test]
    y_true, y_pred, probs = PREDS[(tag, test)]
    wrong = np.where(y_true != y_pred)[0]
    wrong = wrong[np.argsort(-probs[wrong].max(1))][:n]       # erreurs les plus confiantes d'abord
    cols = 4; rows = int(np.ceil(len(wrong) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    for ax in np.ravel(axes): ax.axis("off")
    for ax, i in zip(np.ravel(axes), wrong):
        ax.imshow(Image.open(df.path[i]).convert("RGB"))
        ax.set_title(f"#{i} vrai : {CLASSES[y_true[i]]}\nprédit : {CLASSES[y_pred[i]]} ({probs[i].max():.0%})",
                     fontsize=9)
    plt.tight_layout(); plt.show()

show_errors(best_tag, "pd_test")

✍️ **Hypothèses sur les erreurs** (une ligne par image, à reporter dans le rapport) :

| # image | vrai | prédit | hypothèse (fond, flou, lésion peu visible, plusieurs feuilles, symptômes proches, étiquette douteuse…) |
|---|---|---|---|
| | | | |

## 8. Explicabilité : Grad-CAM

In [ ]:
def gradcam(model, img, class_idx=None, layer_name="last_conv"):
    grad_model = keras.Model(model.inputs, [model.get_layer(layer_name).output, model.outputs[0]])
    x = tf.convert_to_tensor(img[None], tf.float32)
    with tf.GradientTape() as tape:
        conv, preds = grad_model(x, training=False)
        if class_idx is None:
            class_idx = int(tf.argmax(preds[0]))
        score = preds[:, class_idx]
    grads = tape.gradient(score, conv)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam = tf.nn.relu(tf.reduce_sum(conv[0] * weights, axis=-1))
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., None], (IMG_SIZE, IMG_SIZE))[..., 0].numpy()
    return cam, class_idx, float(preds[0, class_idx])

def show_gradcam(model_list, test, n=5, seed=0):
    df = split[test].sample(n, random_state=seed)
    imgs = [load_image(p, 0)[0].numpy().astype("float32") for p in df.path]
    fig, axes = plt.subplots(n, len(model_list) + 1, figsize=(3 * (len(model_list) + 1), 3 * n))
    for r, (img, y) in enumerate(zip(imgs, df.y)):
        axes[r, 0].imshow(img.astype("uint8")); axes[r, 0].set_title(f"vrai : {CLASSES[y]}", fontsize=9)
        for c, m in enumerate(model_list, start=1):
            cam, k, p = gradcam(m, img)
            axes[r, c].imshow(img.astype("uint8")); axes[r, c].imshow(cam, cmap="jet", alpha=0.45)
            axes[r, c].set_title(f"{m.name}\n{CLASSES[k]} ({p:.0%})", fontsize=8)
    for ax in axes.ravel(): ax.axis("off")
    plt.tight_layout(); plt.show()

show_gradcam([e0, e2, BEST_MODEL], "pv_test")
show_gradcam([e0, e2, BEST_MODEL], "pd_test")

In [ ]:
# Mesure chiffrée sur PlantVillage : quelle part de l'attention tombe sur la feuille ?
# (masque de feuille approximatif par saturation, voir leaf_mask)
def attention_on_leaf(model, n=150):
    df = split["pv_test"].sample(n, random_state=SEED)
    ratios = []
    for p in df.path:
        img = load_image(p, 0)[0].numpy().astype("float32")
        cam, _, _ = gradcam(model, img)
        m = leaf_mask(img).numpy()[..., 0]
        ratios.append((cam * m).sum() / (cam.sum() + 1e-8))
    return np.mean(ratios), np.mean([leaf_mask(load_image(p, 0)[0]).numpy().mean() for p in df.path])

for m in [e0, e2, BEST_MODEL]:
    on_leaf, leaf_area = attention_on_leaf(m)
    print(f"{m.name:22s} attention sur la feuille : {on_leaf:.0%} (la feuille occupe {leaf_area:.0%} de l'image)")

✍️ **Analyse** : le modèle regarde-t-il les lésions ou le fond ? L'augmentation de fond change-t-elle la carte d'attention ? Que voit-on sur les images PlantDoc mal classées ?

## 9. Tableau récapitulatif

In [ ]:
final = results_table()
final.to_csv("results/results.csv", index=False)
display(final.sort_values(["test", "rappel_macro"], ascending=[True, False]))
print("Résultats enregistrés dans results/results.csv")

✍️ **Conclusion** (3 à 5 phrases) : quel modèle recommander pour un prototype terrain ? Avec quelles réserves ?

**Pistes bonus** : transfer learning (MobileNetV2, EfficientNetB0), photos locales prises par le groupe, plusieurs graines, seuil de confiance pour « je ne sais pas ».